# Appendix 4 — Human Reasonableness Review

Every label in this study — the benchmark sample's reference labels (Appendix 1) and
the full-corpus production codes (Appendix 3) — is machine-generated. The study's
human-in-the-loop step is a reasonableness review: a human reviewer examines a random sample of
the production outputs, with the decision text alongside, and answers for each code:

> *“Is this coding reasonable, given the text of the decision?”*

The question is deliberately not “is this how I would have coded it?”: many decisions sit on
genuine borderlines (redacted dates, intersecting grounds of persecution, procedural
dispositions) where reasonable coders — human or machine — can disagree. The standard is
deferential by design: a code is judged unreasonable only where it is *not a defensible
reading* of the decision.

**Status.** The review instrument was fixed in advance, and the review was completed by the
principal investigator on 13–14 August 2026. The first cells draw the deterministic (seed-12)
review sample — 100 cases drawn simple-random (unstratified) from the full coded corpus,
excluding every benchmark sample case — and write the labelling project,
`qwen_review_project_random100.json`, that was uploaded to the labelling software: each
document carries the full decision text with the model's codes at the top, and the questions
ask, code by code, whether the coding is reasonable. The completed review is committed as
`data/reasonableness_review.json` and analyzed in the Results section; a supplementary
targeted review of 25 late-claim codes — motivated below — is committed as
`data/reasonableness_review_late25.json` and analyzed in the final section.

AI Disclosure: Generative AI tools were used in preparing these appendices, including for background research, data extraction, coding, drafting and editing (Claude Code, using Fable 5 and 5.5 and Opus 5 and 5.5).

### A second product of this stage: a corpus filter gap, caught and fixed

Human review at the end of the study is not only about scoring the model. While preparing this
stage, the principal investigator hand-reviewed 50 randomly sampled Federal Court decisions
*excluded* by the FC text pre-filter and found a judicial review of a refugee determination
among them — older judgments often name only the “Immigration and Refugee Board”, never the
division — and a targeted follow-up confirmed roughly 5–7% of FC refugee JRs (nearly all
2006–2016) were being missed. (Both the hand review and the follow-up were manual steps that
were not scripted, so the “5–7%” is not reproducible from the committed files; the reproducible
counterpart, computed in Appendix 3, is that the widened decisions the model gated in are 5.2%
of all gated-in FC decisions.) The pre-filter was widened accordingly and the 1,995 newly
captured FC decisions were coded with the identical model, prompt, and settings; Appendix 3
documents the widening.

**What this means for this notebook:** the review below samples from the widened codes
(`data/production_codes.json`, the same corpus Appendices 3 and 5 analyze), so decisions
recovered by the widening are part of the sampling frame.

In [1]:
from pathlib import Path
import json

import pandas as pd

SEED = 12
N_REVIEW = 100        # the principal investigator's reasonableness-review sample
REV = "7ca50f4e7feab57b48223e9252f8aef4d041a29c"   # frozen dataset revision - never change this

DATA = Path("data")   # every file these appendices need ships in this data/ folder

prod = pd.read_json(DATA / "production_codes.json")   # the completed (widened) production run -
                                                    # the corpus Appendices 3 and 5 use
print(f"{len(prod):,} coded decisions available to sample from")

32,462 coded decisions available to sample from


In [2]:
# exclude every case that is in the benchmark sample (matched by production uid where
# recorded, and by (source, citation) for the rest), so the review probes fresh cases
gold = pd.read_json(DATA / "golden_ai_generated_sample.json")
gold_uids = set(gold.prod_uid.dropna())
gold_keys = set(zip(gold.dataset, gold.citation.astype(str)))
pool = prod[~prod.uid.isin(gold_uids)]
pool = pool[[(d, str(c)) not in gold_keys for d, c in zip(pool.dataset, pool.citation)]]
print(f"sampling pool: {len(pool):,} of {len(prod):,} coded decisions "
      f"({len(prod) - len(pool)} overlapping the reference sample excluded)")

# one simple random draw - deliberately NOT stratified, so each source appears in proportion
# to its share of the coded corpus and pooled results read directly at corpus level
review = pool.sample(n=N_REVIEW, random_state=SEED).sort_values("uid", ignore_index=True)
print(f"review sample: {len(review)} cases (deterministic: seed {SEED}, unstratified)")
print(f"model gated out {((review.dataset == 'FC') & (review.jr_refugee != 'a')).sum()} "
      f"of the {(review.dataset == 'FC').sum()} FC cases (only the gate question applies to those)")
review.groupby("dataset").size()

sampling pool: 31,877 of 32,462 coded decisions (585 overlapping the reference sample excluded)
review sample: 100 cases (deterministic: seed 12, unstratified)
model gated out 16 of the 41 FC cases (only the gate question applies to those)


dataset
FC      41
RAD     40
RLLR     3
RPD     16
dtype: int64

In [3]:
BUILD_REVIEW_FILE = False   # True fetches the decision texts from the pinned corpus (~5 min)
                            # and rebuilds the labelling project - run once, before the review

CODE_TEXT = {
    "jr_refugee": {"a": "(a) yes — judicial review of an RPD/RAD refugee determination",
                   "b": "(b) no — some other kind of decision is under review",
                   "c": "(c) unknown from the text"},
    "timing":  {"a": "(a) claim made MORE than one year after the principal claimant's first entry to Canada",
                "b": "(b) claim made one year or LESS after first entry",
                "c": "(c) unclear from the text"},
    "outcome": {"a": "(a) principal claimant successful",
                "b": "(b) principal claimant unsuccessful",
                "c": "(c) unclear from the text"},
    "basis":   {"a": "(a) gender-based persecution alleged (and not SOGIE)",
                "b": "(b) SOGIE-based persecution alleged (and not gender)",
                "c": "(c) both gender and SOGIE allegations",
                "d": "(d) another form of persecution only (no gender/SOGIE allegation)",
                "e": "(e) unclear from the text"},
}

def code_header(r):
    """The model's codes, rendered at the top of each document for the reviewer."""
    lines = ["=== MODEL CODING UNDER REVIEW (Qwen3.5 full-corpus run, Appendix 3) ==="]
    if r.dataset == "FC":
        lines.append("jr_refugee (gate): " + CODE_TEXT["jr_refugee"].get(r.jr_refugee, "(no code)"))
    for q in ["timing", "outcome", "basis"]:
        v = getattr(r, q)
        if isinstance(v, str):
            lines.append(f"{q}: " + CODE_TEXT[q].get(v, f"({v})"))
        else:
            lines.append(f"{q}: not coded — the model gated this case out as not a refugee JR")
    lines.append("Your task: for EACH code above, decide whether it is a REASONABLE reading of the decision below.")
    lines.append("=== DECISION TEXT ===")
    return "\n".join(lines)

REASONABLE_CHOICES = [
    "Yes: the model's code is reasonable",
    "No: the model's code is not reasonable",
    "Not applicable (this question has no code for this case)",
]

QUESTIONS = [
    {"id": "jr_reasonable", "type": "choice",
     "text": "FEDERAL COURT CASES ONLY — is the model's jr_refugee (gate) code reasonable? "
             "(Was it reasonable to code this case as a judicial review of an RPD/RAD refugee "
             "determination — or not one — as the model did?)",
     "choices": REASONABLE_CHOICES,
     "help": "Select 'Not applicable' for RPD, RAD, and RLLR cases (they have no gate code). "
             "The gate counts RPD/RAD merits, vacation/cessation, and reopening/reinstatement "
             "decisions as refugee determinations; PRRA, H&C, visa, detention, removal, "
             "inadmissibility and citizenship matters are not, even if a refugee history is "
             "described as background."},
    {"id": "timing_reasonable", "type": "choice",
     "text": "Is the model's TIMING code reasonable? (Whether the claim was made more than one "
             "year after the principal claimant's FIRST entry to Canada: (a) clearly yes, "
             "(b) clearly no — one year or less, (c) unclear from the text.)",
     "choices": REASONABLE_CHOICES,
     "help": "'First entry' means the first time the principal claimant ever entered Canada in "
             "any status, not the most recent entry. Reasonable inference is allowed (e.g. a "
             "port-of-entry claim, or a fled-to-Canada narrative with a hearing within a year of "
             "arrival, reasonably supports (b)). The question is whether the model's reading is "
             "DEFENSIBLE on this text — not whether it matches the code you would have chosen. "
             "Select 'Not applicable' if the case was gated out and has no timing code."},
    {"id": "outcome_reasonable", "type": "choice",
     "text": "Is the model's OUTCOME code reasonable? (Did the principal claimant succeed with "
             "the claim / appeal / judicial review: (a) yes, (b) no, (c) unclear.)",
     "choices": REASONABLE_CHOICES,
     "help": "Outcome is coded from the principal claimant's perspective (for Minister's "
             "appeals/JRs too). At the RAD, referral back to the RPD counts as success; at the "
             "Federal Court, a JR granted with remittal counts as success. Procedural rulings "
             "with no disposition of the claim are reasonably coded unclear. Select 'Not "
             "applicable' if the case was gated out and has no outcome code."},
    {"id": "basis_reasonable", "type": "choice",
     "text": "Is the model's BASIS (claim type) code reasonable? (What persecution was ALLEGED, "
             "considering the principal claimant and any dependants: (a) gender-based, "
             "(b) SOGIE-based, (c) both, (d) another form only, (e) unclear.)",
     "choices": REASONABLE_CHOICES,
     "help": "Code what was alleged, not what was proven. A claim by a woman is not "
             "automatically gender-based: the alleged harm must itself be gender-based. If "
             "gender-based harm is alleged for any claimant or dependant alongside other "
             "grounds, (a) (or (c) if SOGIE allegations are also present) is the expected code; "
             "(d) applies only when no gender or SOGIE allegation is present. Select 'Not "
             "applicable' if the case was gated out and has no basis code."},
]

INSTRUCTIONS = ("Read the model's codes at the top of the document, then the decision "
                "text. For each coded question, answer whether the model's code is a "
                "REASONABLE reading of the text — not whether it is the code you would "
                "have chosen. Reasonable coders can disagree on borderline cases "
                "(redacted dates, intersecting grounds, procedural dispositions); answer "
                "'No' only where the code is not a defensible reading of the decision. "
                "Where a question has no code (non-FC cases for the gate question; "
                "gated-out FC cases for the other three), select 'Not applicable'.")

if BUILD_REVIEW_FILE:
    from datasets import load_dataset

    def build_project(sample, doc_prefix, out_name, project_id, name, description):
        lookup = sample.copy()
        lookup["orig_idx"] = lookup.uid.str.split("_").str[1].astype(int)
        texts, urls = {}, {}
        for d, sub in lookup.groupby("dataset"):
            df = load_dataset("a2aj/canadian-case-law", data_dir=d, split="train", revision=REV).to_pandas()
            for _, r in sub.iterrows():
                row = df.loc[r.orig_idx]
                texts[r.uid] = str(row["unofficial_text_en"])
                urls[r.uid] = row.get("url_en")
            del df

        documents = []
        for i, r in enumerate(sample.itertuples(), start=1):
            documents.append({
                "id": f"{doc_prefix}-{i:03d}",
                "title": r.citation if isinstance(r.citation, str) and r.citation else r.uid,
                "meta": {
                    "citation": r.citation if isinstance(r.citation, str) else r.uid,
                    "url": urls[r.uid],
                    "tribunal": r.dataset,
                    "uid": r.uid,
                    "document_date": r.document_date,
                    "qwen_jr_refugee": r.jr_refugee if isinstance(r.jr_refugee, str) else None,
                    "qwen_timing": r.timing if isinstance(r.timing, str) else None,
                    "qwen_outcome": r.outcome if isinstance(r.outcome, str) else None,
                    "qwen_basis": r.basis if isinstance(r.basis, str) else None,
                },
                "text": code_header(r) + "\n" + texts[r.uid],
            })

        project = {"id": project_id, "name": name, "description": description,
                   "instructions": INSTRUCTIONS, "labels_per_doc": 1, "active": True,
                   "questions": QUESTIONS, "documents": documents}
        out = DATA / out_name
        out.write_text(json.dumps(project, ensure_ascii=False, indent=2), encoding="utf-8")
        print(f"wrote {out}  ({out.stat().st_size / 1e6:.1f} MB, {len(documents)} documents)")

    build_project(review, "r100", "qwen_review_project_random100.json",
                  "random_100_reasonablness_review",
                  "Reasonableness review — random 100 (unstratified)",
                  "A reasonableness-review sample of 100 decisions drawn simple-random "
                  "(seed 12, unstratified) from the widened full-corpus model coding — every "
                  "decision passing the text pre-filter, excluding the benchmark "
                  "sample — so sources appear in proportion to the corpus. Each document "
                  "shows the model's codes at the top, followed by the full decision text. "
                  "For each code, assess whether it is a reasonable reading of the decision.")
else:
    print("BUILD_REVIEW_FILE is False - not fetching texts / writing the project file")

BUILD_REVIEW_FILE is False - not fetching texts / writing the project file


**Notes on the instrument.** The sample is deterministic (seed 12), excludes every benchmark
case (so the review probes fresh, never-validated codings), and was fixed
before any results were seen. It is deliberately unstratified: sources appear in proportion to
their corpus shares, so pooled results read directly as corpus-level estimates. Each
case's `uid` links the review back to the production codes of Appendix 3. The model's codes are
shown to the reviewer by design — assessing the reasonableness of a given coding requires
seeing it.

## Results

The completed review (a single reviewer, one label per document) is committed as
`data/reasonableness_review.json` — the labelling-software export with the embedded decision
texts (already public via the pinned corpus) and the reviewer's email address removed; nothing
else is altered. Each of the 100 cases carries four answers (`jr_reasonable`,
`timing_reasonable`, `outcome_reasonable`, `basis_reasonable`), each one of *Yes* (the code is
reasonable), *No* (it is not), or *Not applicable* (no code to review: the gate question for
non-FC cases, the three main questions for gated-out FC cases), plus an optional free-text
comment.

The cells below load the export, verify it covers exactly the sample drawn above, compute the
share of codes judged reasonable per question, and list every code judged *not*
reasonable.

In [4]:
RESULTS_FILE = DATA / "reasonableness_review.json"

ANSWER = {"Yes: the model's code is reasonable": "yes",
          "No: the model's code is not reasonable": "no",
          "Not applicable (this question has no code for this case)": "n/a"}

labels = json.loads(RESULTS_FILE.read_text(encoding="utf-8"))["labels"]
rr = pd.DataFrame([{
    "doc_id": l["doc_id"], "uid": l["doc_meta"]["uid"], "source": l["doc_meta"]["tribunal"],
    "citation": l["doc_meta"]["citation"],
    # the model codes under review (also embedded in the document metadata at build time)
    "jr_refugee": l["doc_meta"]["qwen_jr_refugee"], "timing": l["doc_meta"]["qwen_timing"],
    "outcome": l["doc_meta"]["qwen_outcome"], "basis": l["doc_meta"]["qwen_basis"],
    # the reviewer's four answers, mapped to yes / no / n/a
    **{f"{q}_ok": ANSWER[l["answers"][f"{q}_reasonable"]]
       for q in ["jr", "timing", "outcome", "basis"]},
    "comments": l["comments"],
} for l in labels])

# integrity: the export covers exactly the deterministic sample drawn above, the gate question
# was answered only for FC cases, and the main questions are n/a exactly where the model gated
# the case out (jr_refugee != a)
assert len(rr) == N_REVIEW and set(rr.uid) == set(review.uid)
assert rr.doc_id.str.match(r"r100-\d{3}$").all()
assert (rr.loc[rr.source != "FC", "jr_ok"] == "n/a").all()
gated_out = set(rr.loc[(rr.source == "FC") & (rr.jr_refugee != "a"), "uid"])
for q in ["timing", "outcome", "basis"]:
    assert set(rr.loc[rr[f"{q}_ok"] == "n/a", "uid"]) == gated_out

print(f"{len(rr)} reviewed cases — exactly the seed-12 sample; "
      f"{len(gated_out)} FC cases gated out by the model (no main codes to review); "
      f"{rr.comments.notna().sum()} cases carry a reviewer comment")

100 reviewed cases — exactly the seed-12 sample; 16 FC cases gated out by the model (no main codes to review); 32 cases carry a reviewer comment


In [5]:
from scipy.stats import binomtest

MAIN_QUESTIONS = ["timing", "outcome", "basis"]

def exact_ci(ok, n):
    """Clopper-Pearson (exact binomial) 95% interval for ok successes in n trials."""
    ci = binomtest(ok, n).proportion_ci(confidence_level=0.95, method="exact")
    return f"[{ci.low:.3f}, {ci.high:.3f}]"

def rate(sub, q):
    """(codes judged reasonable, codes reviewed) for question q, ignoring n/a."""
    coded = sub[sub[f"{q}_ok"] != "n/a"]
    return (coded[f"{q}_ok"] == "yes").sum(), len(coded)

# headline: share of codes judged reasonable, per question. The draw is unstratified, so these
# pooled shares are directly corpus-representative (no reweighting needed)
headline = pd.DataFrame([
    {"question": name, "codes reviewed": n, "judged reasonable": ok, "share": round(ok / n, 3),
     "95% CI (exact)": exact_ci(ok, n)}
    for q, name in [("jr", "jr_refugee (gate, FC only)"), ("timing", "timing"),
                    ("outcome", "outcome"), ("basis", "basis")]
    for ok, n in [rate(rr, q)]])
display(headline)

,question,codes reviewed,judged reasonable,share,95% CI (exact)
0,"jr_refugee (gate, FC only)",41,41,1.000,"[0.914, 1.000]"
1,timing,84,84,1.000,"[0.957, 1.000]"
2,outcome,84,84,1.000,"[0.957, 1.000]"
3,basis,84,83,0.988,"[0.935, 1.000]"


In [6]:
# all codes pooled across the four questions, with the exact interval
ok_all, n_all = (sum(x) for x in zip(*(rate(rr, q) for q in ["jr", "timing", "outcome", "basis"])))
print(f"all codes reviewed: {ok_all}/{n_all} = {ok_all / n_all:.3f}, 95% CI {exact_ci(ok_all, n_all)}")

all codes reviewed: 292/293 = 0.997, 95% CI [0.981, 1.000]


In [7]:
# every code the reviewer judged NOT reasonable, with the model's code and the reviewer's comment
flagged = pd.concat([
    rr[rr[f"{q}_ok"] == "no"].assign(question=q, model_code=rr.loc[rr[f"{q}_ok"] == "no", q])
    for q in MAIN_QUESTIONS], ignore_index=True)
with pd.option_context("display.max_colwidth", None):
    display(flagged[["question", "doc_id", "source", "citation", "model_code", "comments"]])

,question,doc_id,source,citation,model_code,comments
0,basis,r100-014,FC,2025 FC 533,d,"Basis should be (a), while gender not main claim type, it does arise (see e.g. 28). Note that this is a section 108 case."


### What the review found

**Overall, 292 of the 293 codes reviewed (99.7%) were judged reasonable** (exact 95% interval
0.981–1). Every gate code (41/41; interval 0.914–1), every timing code (84/84; interval
0.957–1), and every outcome code (84/84) passed; basis passed in 83/84 cases (0.988; interval
0.935–1). Because the sample is unstratified, these pooled shares are direct corpus-level
estimates.

**No timing code was judged unreasonable — in particular, none coded a claim as late without
basis.** Timing is the question the validation identified as hardest (Appendix 2: agreement
≈0.91, against ≈0.96–1.00 for the others), and it is where the deferential standard does real
work. The reviewer's comments record a series of borderline timing calls — typically an entry
date but no claim date — where the reviewer would personally have coded *(c) unclear* but
judged the model's *(b) timely* a defensible inference, usually because a decision otherwise
attentive to credibility and delay says nothing about a delay in claiming; eleven of the 32
comments are of this kind (a manual reading of the comments — the count is not scripted), and
several state that *(b)* and *(c)* are both reasonable readings of the same text. What matters for the study is the direction: the inference resolves silence
to *timely*, never to *late*, so to the extent the model over-resolves, it under-counts late
claims — the same conservative direction as the validation in Appendix 2 (late-claim precision
≈0.95).

**The one basis miss is an under-count**: a *(d) other persecution* code where a gender-based
allegation does arise in the decision (a cessation-related case). Several further comments
discuss borderline basis calls — claims with gender elements the decision-maker did not treat
as the frame — and accept the model's code under the instrument's alleged-basis rule. One of
these (r100-041, *Zhang v Canada (MCI)*, 2019 FC 870) is a family-planning claim by a male
principal claimant fearing forced sterilization, which the decision does not frame in gender
terms; the reviewer judged the model's *(a) gender* reasonable because the instrument
expressly treats forced sterilization and other reproductive coercion as gender-based
persecution. The only miss runs against, not for, the study's finding.

**Implication for Appendix 5.** The human review supports relying on the full-corpus codes:
unreasonable codes are rare (1 of 293, on basis, and an under-count), no timing, outcome, or
gate code was judged unreasonable, and the borderline behaviour the comments document —
resolving delay-silence to *timely* — works *against*, not for, the study's finding that late
claims are common and disproportionately gender/SOGIE-based.

## Supplementary review: 25 late-claim codes

The random-100 review is corpus-representative, but it contains only 8 decisions the model
coded as late claims: late claims are central to the Bill C-12 analysis yet only about 8% of
coded decisions. Since the headline findings all run through the late-claim code
(timing = *(a)*), a supplementary targeted review was undertaken: 25 decisions drawn at random
(seed 12) from all late-coded decisions, excluding the benchmark sample (the draw
happens to share no cases with the random 100). The instrument
(`qwen_review_project_late25.json`) is identical to the main one, and the review was completed
by the same reviewer; the export is committed as `data/reasonableness_review_late25.json`.
Because the sample conditions on the model saying *late*, it measures one thing in depth — is
the late-claim code defensible where it matters most? It is a precision check, not a
corpus-representative sample, so its gate, outcome, and basis rates describe a group selected
on the late code and should not be pooled with the random 100's rates for those questions. The
late-claim code itself is the exception: the 8 late-coded cases in the random 100 (a simple
random sample of the pool, so its late-coded cases are a simple random sample of the pool's
late-coded decisions) and the 25 targeted cases are both random draws from the model's
late-coded decisions, so the share of late codes upheld can be pooled across the two samples,
as the last lines of the cell below do (32/33).

In [8]:
# the late-claim draw: 25 at random from every decision the model coded timing = (a)
late25 = pool[pool.timing == "a"].sample(n=25, random_state=SEED).sort_values("uid", ignore_index=True)
print(f"late-claim pool: {(pool.timing == 'a').sum():,} of {len(pool):,} coded decisions "
      f"({(pool.timing == 'a').mean():.1%}); drew {len(late25)}")
print("by source:", late25.dataset.value_counts().to_dict(),
      "| overlap with the random-100 sample:", len(set(late25.uid) & set(review.uid)))

if BUILD_REVIEW_FILE:
    build_project(late25, "lc25", "qwen_review_project_late25.json",
                  "late_claims_25_reasonablness_review",
                  "Reasonableness review — 25 late-claim codes",
                  "A targeted reasonableness-review sample of 25 decisions the model coded as "
                  "LATE claims (timing = (a), claim made more than one year after first "
                  "entry), drawn at random (seed 12) from all late-coded decisions in the "
                  "widened full-corpus model coding, excluding the benchmark "
                  "sample. Late claims are the study's key group but are only ~8% of coded "
                  "decisions, so a general random sample contains few of them; this sample "
                  "reviews the late-claim code in depth. Each document shows the model's "
                  "codes at the top, followed by the full decision text. For each code, "
                  "assess whether it is a reasonable reading of the decision.")

# load the completed supplementary review and verify it covers exactly this draw
labels25 = json.loads((DATA / "reasonableness_review_late25.json").read_text(encoding="utf-8"))["labels"]
rr25 = pd.DataFrame([{
    "doc_id": l["doc_id"], "uid": l["doc_meta"]["uid"], "source": l["doc_meta"]["tribunal"],
    "citation": l["doc_meta"]["citation"],
    "jr_refugee": l["doc_meta"]["qwen_jr_refugee"], "timing": l["doc_meta"]["qwen_timing"],
    "outcome": l["doc_meta"]["qwen_outcome"], "basis": l["doc_meta"]["qwen_basis"],
    **{f"{q}_ok": ANSWER[l["answers"][f"{q}_reasonable"]]
       for q in ["jr", "timing", "outcome", "basis"]},
    "comments": l["comments"],
} for l in labels25])

assert len(rr25) == 25 and set(rr25.uid) == set(late25.uid)
assert rr25.doc_id.str.match(r"lc25-\d{3}$").all()
assert (rr25.timing == "a").all()                          # by construction: late codes only
assert (rr25.loc[rr25.source != "FC", "jr_ok"] == "n/a").all()
assert not (rr25[["timing_ok", "outcome_ok", "basis_ok"]] == "n/a").any().any()

headline25 = pd.DataFrame([
    {"question": name, "codes reviewed": n, "judged reasonable": ok, "share": round(ok / n, 3),
     "95% CI (exact)": exact_ci(ok, n)}
    for q, name in [("jr", "jr_refugee (gate, FC only)"), ("timing", "timing (all code a)"),
                    ("outcome", "outcome"), ("basis", "basis")]
    for ok, n in [rate(rr25, q)]])
display(headline25)

# every code judged not reasonable, with the reviewer's comment
flagged25 = pd.concat([
    rr25[rr25[f"{q}_ok"] == "no"].assign(question=q, model_code=rr25.loc[rr25[f"{q}_ok"] == "no", q])
    for q in ["timing", "outcome", "basis"]], ignore_index=True)
with pd.option_context("display.max_colwidth", None):
    display(flagged25[["question", "doc_id", "source", "citation", "model_code", "comments"]])

# the study's key number: late-claim codes upheld, here and in the random 100
ok25, n25 = rate(rr25, "timing")
late_rand = rr[(rr.timing == "a") & (rr.timing_ok != "n/a")]
print(f"late-claim codes judged reasonable — targeted sample: {ok25}/{n25}; "
      f"random 100: {(late_rand.timing_ok == 'yes').sum()}/{len(late_rand)}; "
      f"combined: {ok25 + (late_rand.timing_ok == 'yes').sum()}/{n25 + len(late_rand)}")

late-claim pool: 2,414 of 31,877 coded decisions (7.6%); drew 25
by source: {'RAD': 13, 'FC': 8, 'RLLR': 2, 'RPD': 2} | overlap with the random-100 sample: 0


,question,codes reviewed,judged reasonable,share,95% CI (exact)
0,"jr_refugee (gate, FC only)",8,8,1.00,"[0.631, 1.000]"
1,timing (all code a),25,24,0.96,"[0.796, 0.999]"
2,outcome,25,25,1.00,"[0.863, 1.000]"
3,basis,25,25,1.00,"[0.863, 1.000]"


,question,doc_id,source,citation,model_code,comments
0,timing,lc25-019,RAD,MB7-04078,a,"timing category is borderline. Depends on how one interprets this sentence: ""The RPD found that the Appellant's delay in claiming refugee protection indicated a lack of legitimate fear of persecution because the Appellant claims to have feared the authorities in Syria since 2011 yet traveled to the United States, Canada and Italy since 2011 without claiming asylum."" He arrived most recently in 2016 and most likely claimed in 2016 -- but the other sentence suggests another visit to Canada. The most likely timeline for another visit to Canada at some point from 2011 on, would result in 1 year +. However, if the visit came at the end of the range, then it would not be. I went back and forth on this one a lot, but I think (c) is really the only viable option, even if I can understand why the model said (a)."


late-claim codes judged reasonable — targeted sample: 24/25; random 100: 8/8; combined: 32/33


**Why 41 late codes are excluded here, not the paper's 39.** The late-claim pool above is
2,414, while Appendix 5 counts 2,455 late claims; the difference is the 41 benchmark cases
the production run coded late, removed with the rest of the benchmark sample. Table 2's 39 is a
different count — the validation run's (Appendix 2) late codes for the same benchmark cases. At
temperature 0 the two runs still differ on a few timing calls (Appendix 3), and one of the 39
falls outside the 2006–2025 corpus. The cell below shows the reconciliation.

In [9]:
# reconcile the 41 late codes excluded here with the 39 behind Table 2: link each benchmark
# case to its production record (as Appendix 3 does) and compare the two runs' timing codes
val = pd.read_json(DATA / "golden_predictions_qwen.json")       # Appendix 2: validation-run codes
uid_by_key = dict(zip(zip(prod.dataset, prod.citation.astype(str)), prod.uid))
bench = gold[["case_id", "dataset", "citation", "prod_uid"]].merge(val, on=["case_id", "dataset"])
bench["uid"] = [u if isinstance(u, str) else uid_by_key.get((d, str(c)))
                for u, d, c in zip(bench.prod_uid, bench.dataset, bench.citation)]
twice = bench.dropna(subset=["uid"]).merge(prod[["uid", "timing"]], on="uid")
v, p = twice.pred_timing == "a", twice.timing == "a"
print(f"late codes: {(prod.timing == 'a').sum():,} in the full corpus (Appendix 5) - "
      f"{(pool.timing == 'a').sum():,} in this pool = {p.sum()} benchmark cases coded late by the production run")
print(f"validation run (Table 2): {(bench.pred_timing == 'a').sum()} benchmark cases coded late, "
      f"{(bench.uid.isna() & (bench.pred_timing == 'a')).sum()} of them outside the corpus "
      f"-> {v.sum()} of the {len(twice)} cases coded by both runs")
print(f"of those {len(twice)}: {(v & p).sum()} late in both runs, {(v & ~p).sum()} late only in the validation run, "
      f"{(p & ~v).sum()} late only in the production run")

late codes: 2,455 in the full corpus (Appendix 5) - 2,414 in this pool = 41 benchmark cases coded late by the production run
validation run (Table 2): 39 benchmark cases coded late, 1 of them outside the corpus -> 38 of the 585 cases coded by both runs
of those 585: 36 late in both runs, 2 late only in the validation run, 5 late only in the production run


In [10]:
# exact 95% interval for the late-claim code pooled across the two samples, and all codes in the targeted sample
ok_rand, n_rand = (late_rand.timing_ok == "yes").sum(), len(late_rand)
print(f"late-claim codes judged reasonable, combined: {ok25 + ok_rand}/{n25 + n_rand} = "
      f"{(ok25 + ok_rand) / (n25 + n_rand):.3f}, 95% CI {exact_ci(ok25 + ok_rand, n25 + n_rand)}")
ok_all25, n_all25 = (sum(x) for x in zip(*(rate(rr25, q) for q in ["jr", "timing", "outcome", "basis"])))
print(f"all codes in the targeted sample: {ok_all25}/{n_all25} = {ok_all25 / n_all25:.3f}, 95% CI {exact_ci(ok_all25, n_all25)}")

late-claim codes judged reasonable, combined: 32/33 = 0.970, 95% CI [0.842, 0.999]
all codes in the targeted sample: 82/83 = 0.988, 95% CI [0.935, 1.000]


### The paper's Table 3

The two reviews side by side, in the form the paper reports them: codes judged reasonable /
codes reviewed, per question, with the late-claim code (timing = a) as its own row. The
targeted sample conditions on the model coding a claim late, so the two columns are not
pooled except on that row (32/33, above).


In [11]:
def ok_n(sub, q):
    return "{}/{}".format(*rate(sub, q))

table3 = pd.DataFrame({
    "random sample (100 decisions)":
        [ok_n(rr, q) for q in ["jr", "timing", "outcome", "basis"]] + [f"{ok_rand}/{n_rand}"],
    "targeted sample (25 late-claim decisions)":
        [ok_n(rr25, q) for q in ["jr", "timing", "outcome", "basis"]] + [f"{ok25}/{n25}"],
}, index=["Federal Court screen", "Timing", "Outcome", "Basis", "Late-claim categorization"])
display(table3)


,random sample (100 decisions),targeted sample (25 late-claim decisions)
Federal Court screen,41/41,8/8
Timing,84/84,24/25
Outcome,84/84,25/25
Basis,83/84,25/25
Late-claim categorization,8/8,24/25


### What the supplementary review found

**82 of the 83 codes reviewed (99%) were judged reasonable — including 24 of the 25 late-claim
timing codes (exact 95% interval 0.796–0.999).** Every gate (8/8), outcome (25/25), and basis (25/25) code passed. The one
timing code judged not reasonable is a RAD decision the reviewer found genuinely ambiguous —
the comment calls the category “borderline”, turning on how a single sentence about the
claimant's travel history is read — a defensible-versus-defensible disagreement rather than a
clear over-count.

**Combined across the two review samples, 32 of 33 late-claim codes (97%; exact 95% interval
0.842–0.999) were upheld.** This
is the direct human check on the study's key measurement: when the model says a claim was made
more than a year after first entry, a deferential human review agrees the text supports it in
all but one borderline case. Together with the validation (Appendix 2: late-claim precision
≈0.95) and the conservative timing direction documented above (silence resolved to *timely*,
not *late*), the late-claim group at the centre of Appendix 5 rests on codes that human review
consistently sustains.